# History-rung benchmark: k-nearest neighbours (GPU)

The kNN member of the history-rung benchmark, on the rows, folds, rungs and protocol of `13_History_Models`. It runs on the RAPIDS kernel
(`knn_and_svr_gpu_env`) like `06_kNN`, with the same pipeline (standardised inputs, cuML `KNeighborsRegressor`) and the same search space
(number of neighbours, uniform or distance weights, Manhattan or Euclidean metric). The search is one Bayesian search of 20 candidates on rung H4
on all training rows, scored on the five expanding-window folds; the chosen configuration is then fitted at every benchmark rung.

In [1]:
import json
import time
import numpy as np
import pandas as pd
import cupy as cp
from scipy.stats import norm
from sklearn.mixture import GaussianMixture
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from skopt import BayesSearchCV
from skopt.space import Categorical, Integer
from cuml.neighbors import KNeighborsRegressor

RANDOM_STATE = 42
TARGETS = (0.90, 0.95, 0.99)


In [2]:
train = pd.read_csv("Data_Files/history_train.csv")
test = pd.read_csv("Data_Files/history_test.csv")
rungs = json.load(open("Data_Files/history_rungs.json"))
d = pd.concat([train, test], ignore_index=True)
d["t"] = pd.to_datetime(d["t"], utc=True)
d = d.sort_values("t").reset_index(drop=True)
fold = d["fold"].to_numpy()
H = {k.split()[0]: v for k, v in rungs.items()}
BENCH = ["H1", "H4", "H6", "H7"]

ok = (d["y"].notna() & d[H["H7"]].notna().all(axis=1)).to_numpy()
is_test = fold == 9
cal = ok & (fold >= 0) & (fold < 9)
ev = ok & is_test
PL, a, y = d["PL"].to_numpy(), d["a"].to_numpy(), d["y"].to_numpy()
X_all = d[H["H7"]].copy()
X_all = X_all.fillna(X_all[ok & (fold < 9)].median())
print(f"rows: training {int((ok & (fold < 9)).sum()):,} | calibration (folds 0-4) {int(cal.sum()):,} | hold-out {int(ev.sum()):,} of {int(is_test.sum()):,}")


rows: training 2,126,679 | calibration (folds 0-4) 1,773,285 | hold-out 531,983 of 532,055


In [3]:
def rmse(e):
    return float(np.sqrt(np.mean(np.square(e))))

def mixture_quantile(w, mu, sd, q):
    lo, hi = float((mu - 10 * sd).min()), float((mu + 10 * sd).max())
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if float((w * norm.cdf((mid - mu) / sd)).sum()) < q else (lo, mid)
    return 0.5 * (lo + hi)

def margins(e_cal, e_test):
    g = GaussianMixture(n_components=3, covariance_type="full", random_state=RANDOM_STATE).fit(e_cal.reshape(-1, 1))
    w, mu, sd = g.weights_, g.means_.ravel(), np.sqrt(g.covariances_.reshape(-1))
    out = {}
    for rho in TARGETS:
        emp = float(np.quantile(e_cal, rho))
        fm = max(emp, mixture_quantile(w, mu, sd, rho)) if rho >= 0.98 else emp
        out[f"margin {rho:.2f}"], out[f"reached {rho:.2f}"] = fm, float((e_test <= fm).mean())
    return out

def predict(model, X):
    return np.asarray(cp.asnumpy(cp.asarray(model.predict(X))), dtype=float).ravel()

def oof_and_test(make, cols):
    X = X_all[cols].to_numpy(dtype=np.float32)
    p = np.full(len(d), np.nan)
    for k in range(5):
        tr, va = ok & (fold < k), ok & (fold == k)
        p[va] = predict(make().fit(X[tr], y[tr]), X[va])
    tr = ok & (fold < 9)
    model = make().fit(X[tr], y[tr])
    p[ev] = predict(model, X[ev])
    return p, model

def save_residuals(name, p):
    for split, mask in (("oof", cal), ("test", ev)):
        pd.DataFrame({"model": name, "split": split, "row_id": np.flatnonzero(mask), "fold": fold[mask], "time": d["t"].to_numpy()[mask],
                      "device_id": d["device_id"].to_numpy()[mask], "PL_true": PL[mask], "PL_pred": (a + p)[mask], "resid_db": (y - p)[mask]}
                     ).to_csv(f"Residuals/residuals_{name}_{split}.csv", index=False)

def evaluate(p):
    e = y - p
    row = {"hold-out RMSE": rmse(e[ev]), "hold-out MAE": float(np.abs(e[ev]).mean()), "OOF RMSE": rmse(e[cal])}
    row.update(margins(e[cal], e[ev]))
    return row


In [4]:
# One Bayesian search on rung H4, all training rows, the five expanding-window folds
def make_knn(**params):
    return Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsRegressor(**params))])

rows_train = np.flatnonzero(ok & (fold < 9))
X4, y4, fold4 = X_all[H["H4"]].to_numpy(dtype=np.float32)[rows_train], y[rows_train], fold[rows_train]
cv = [(np.flatnonzero(fold4 < k), np.flatnonzero(fold4 == k)) for k in range(5)]
space = {"knn__n_neighbors": Integer(20, 1000, prior="log-uniform"), "knn__weights": Categorical(["uniform", "distance"]), "knn__metric": Categorical(["manhattan", "euclidean"])}

t0 = time.time()
bs = BayesSearchCV(make_knn(), space, n_iter=20, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=1, random_state=RANDOM_STATE, refit=False)
bs.fit(X4, y4, callback=lambda res: cp.get_default_memory_pool().free_all_blocks())
pd.DataFrame(bs.cv_results_).to_csv("CV_Results/history_KNN_search.csv", index=False)
best = {k.replace("knn__", ""): v for k, v in bs.best_params_.items()}
json.dump(best, open("CV_Results/history_KNN_best_params.json", "w"), indent=1, default=str)
print(f"kNN: best CV RMSE {-bs.best_score_:.3f} dB after {time.time() - t0:.0f} s -> {best}")


kNN: best CV RMSE 2.939 dB after 984 s -> {'metric': 'manhattan', 'n_neighbors': 45, 'weights': 'distance'}


In [5]:
results = {}
for rung in BENCH:
    t0 = time.time()
    p, _ = oof_and_test(lambda: make_knn(**best), H[rung])
    results[("KNN", rung)] = evaluate(p)
    save_residuals(f"KNN_{rung}", p)
    print(f"KNN {rung}: hold-out RMSE {results[('KNN', rung)]['hold-out RMSE']:.3f} dB, margin 0.99 {results[('KNN', rung)]['margin 0.99']:.2f} dB  ({time.time() - t0:.0f} s)")
table = pd.DataFrame(results).T
table.index.names = ["family", "rung"]
table.to_csv("CV_Results/history_benchmark_KNN.csv")
table.round(3)


KNN H1: hold-out RMSE 2.460 dB, margin 0.99 11.22 dB  (37 s)


KNN H4: hold-out RMSE 2.157 dB, margin 0.99 10.74 dB  (40 s)


KNN H6: hold-out RMSE 2.160 dB, margin 0.99 10.95 dB  (52 s)


KNN H7: hold-out RMSE 2.195 dB, margin 0.99 11.08 dB  (52 s)


hold-out RMSE  hold-out MAE  OOF RMSE  margin 0.90  reached 0.90  \
family rung                                                                     
KNN    H1            2.460         1.857     3.206        3.354         0.921   
       H4            2.157         1.607     2.951        2.887         0.930   
       H6            2.160         1.607     2.974        2.970         0.930   
       H7            2.195         1.637     3.003        3.024         0.930   

             margin 0.95  reached 0.95  margin 0.99  reached 0.99  
family rung                                                        
KNN    H1          4.760         0.970       11.218         0.999  
       H4          4.154         0.976       10.737         0.999  
       H6          4.255         0.976       10.950         0.999  
       H7          4.321         0.975       11.085         0.999

In [6]:
# A link the model has never seen (rung H4)
dev = d["device_id"].to_numpy()
X4_all = X_all[H["H4"]].to_numpy(dtype=np.float32)
unseen = {}
for link in sorted(np.unique(dev)):
    other, this = dev != link, dev == link
    fit_m, cal_m, test_m = ok & other & (fold >= 0) & (fold < 4), cal & other & (fold == 4), ev & this
    m = make_knn(**best).fit(X4_all[fit_m], y[fit_m])
    e_cal, e_test = y[cal_m] - predict(m, X4_all[cal_m]), y[test_m] - predict(m, X4_all[test_m])
    fm = float(np.quantile(e_cal, 0.99))
    unseen[("KNN", link)] = {"RMSE on the unseen link": rmse(e_test), "margin 0.99 from five links": fm, "reached on the unseen link": float((e_test <= fm).mean())}
unseen_df = pd.DataFrame(unseen).T
unseen_df.index.names = ["family", "link"]
unseen_df.to_csv("CV_Results/history_unseen_link_KNN.csv")
unseen_df.round(3)


RMSE on the unseen link  margin 0.99 from five links  \
family link                                                         
KNN    ED0                     2.275                        7.467   
       ED1                     2.394                        7.438   
       ED2                     2.492                        7.142   
       ED3                     2.154                        7.320   
       ED4                     2.187                        7.484   
       ED5                     2.202                        7.434   

             reached on the unseen link  
family link                              
KNN    ED0                        0.997  
       ED1                        0.998  
       ED2                        0.994  
       ED3                        0.999  
       ED4                        0.998  
       ED5                        0.998